# CoAD reproduction — TSB-AD, 64 series

Batch runs of CoAD and the 7 data-mining baselines on the 64 TSB-AD series of Table 1 (KDD 2026 paper).
Self-contained: run 23.1 to 23.5 in every new runtime, adjust 23.6, then run 23.7. Re-run 23.7 to continue after a disconnect.
23.8 shows progress and the comparison with the paper at any time.

Setup, EDA and the single-series pilot are in `CoAD_reproduction.ipynb`; explanations in `huong-dan-reproduce-coad.md`, section 23.

## 23.1. Session setup

Mount Drive (choose the account that owns `CoAD_project`) and check out the commit saved in `env/commit.txt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, re, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

STORE = Path('/content/drive/MyDrive/CoAD_project')
REPO = Path('/content/CoAD')
commit = (STORE / 'env' / 'commit.txt').read_text(encoding='utf-8').strip()

if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/richard-tang199/CoAD.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', '-q', commit], cwd=REPO, check=True)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Commit: 3286813619091eeebec04b3c9d73c837bdc4469e


## 23.2. Install dependencies (once per runtime)

Installs the original `requirements.txt` of the pinned commit; full log in `env/install_experiments.log`. No restart needed: every run is a new Python process.

In [ ]:
# Packages imported by baseline_compare.py, the evaluator and CoAD; checked in a fresh process
CHECK_IMPORTS = 'import arch, stumpy, tslearn, sktime, torch_geometric, statsmodels, numba'

def deps_ok():
    return subprocess.run([sys.executable, '-c', CHECK_IMPORTS], capture_output=True).returncode == 0

if not deps_ok():
    install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')],
                             capture_output=True, text=True)
    (STORE / 'env' / 'install_experiments.log').write_text(install.stdout + install.stderr, encoding='utf-8')
    print('pip exit code:', install.returncode)
    assert install.returncode == 0, 'Installation failed; inspect env/install_experiments.log'
assert deps_ok(), 'Some packages still cannot be imported; inspect env/install_experiments.log'

# Use the current tslearn (about 3x faster than 0.6.3); SAND's short last batch is handled in 23.3
tslearn_version = subprocess.check_output([sys.executable, '-c', 'import tslearn; print(tslearn.__version__)'], text=True).strip()
if tslearn_version.startswith('0.6.'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--no-deps', 'tslearn'], check=True)
    tslearn_version = subprocess.check_output([sys.executable, '-c', 'import tslearn; print(tslearn.__version__)'], text=True).strip()
print('All required packages import in this runtime | tslearn', tslearn_version)

All required packages import in this runtime | tslearn 0.10.0


## 23.3. Data and `baseline_compare.py` imports

Extracts TSB-AD-U from the archive already on Drive (870 CSVs; group IDs are positions in the sorted list) and declares the three imports `baseline_compare.py` uses.

In [ ]:
import zipfile
archive = STORE / 'data_source' / 'TSB-AD-U.zip'
RAW = REPO / 'dataset' / 'TSB-AD' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for m in z.namelist():
        if m.startswith('TSB-AD-U/') and m.endswith('.csv') and '..' not in m:
            target = RAW / Path(m).name
            if not target.exists():
                with z.open(m) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
RAW_FILES = sorted(p.name for p in RAW.iterdir())
assert len(RAW_FILES) == 870, f'raw/ must contain exactly 870 CSVs, found {len(RAW_FILES)}'

baseline_script = REPO / 'baseline_compare.py'
required_imports = 'import os\nimport numpy as np\nimport torch\n'
script_source = baseline_script.read_text(encoding='utf-8')
if not script_source.startswith(required_imports):
    baseline_script.write_text(required_imports + script_source, encoding='utf-8')

# NumPy 2.0 removed the aliases np.Inf / np.NaN; the baselines use them (same values as np.inf / np.nan)
NUMPY2_RENAMES = {'np.Inf': 'np.inf', 'np.NaN': 'np.nan'}
for rel in ['baseline/sand.py', 'baseline/feature.py']:
    path = REPO / rel
    source = path.read_text(encoding='utf-8')
    for old, new in NUMPY2_RENAMES.items():
        source = source.replace(old, new)
    path.write_text(source, encoding='utf-8')
# SAND clusters each online batch into k = 6 groups; the last batch of some series has fewer than 6
# subsequences and tslearn >= 0.8 then fails. Use at most as many clusters as subsequences in that batch.
sand_file = REPO / 'baseline' / 'sand.py'
sand_source = sand_file.read_text(encoding='utf-8')
old_line = 'ks = KShape(n_clusters=self.k, verbose=False)'
new_line = 'ks = KShape(n_clusters=min(self.k, len(all_subsequences)), verbose=False)'
if old_line in sand_source:
    sand_file.write_text(sand_source.replace(old_line, new_line), encoding='utf-8')
assert new_line in sand_file.read_text(encoding='utf-8'), 'SAND patch not applied'

diff_text = subprocess.check_output(['git', 'diff', '--', 'baseline_compare.py', 'baseline/sand.py', 'baseline/feature.py'],
                                    cwd=REPO, text=True)
(STORE / 'env' / 'baseline_compat.diff').write_text(diff_text, encoding='utf-8')
changed = sum(1 for line in diff_text.splitlines() if line.startswith('+') and not line.startswith('+++'))
print('Data ready:', len(RAW_FILES), 'files |', changed, 'lines added/changed in baseline code (env/baseline_compat.diff)')

Data ready: 870 files | 9 lines added/changed in baseline code (env/baseline_compat.diff)


## 23.4. One-run helper

Each run gets `runs/tsbad64/<time>_<method>_TSB-AD_<group>/` with `command.json`, `console.log`, `artifacts/`. One row per run is appended to `results/tsbad64_results.csv` right away, so finished runs survive a disconnect. A failing run is recorded as `failed` (last log lines in `error`) instead of stopping the batch.

In [ ]:
RESULTS_DIR = STORE / 'results'   # each session appends to its own file, set in 23.6
RUNS = STORE / 'runs' / 'tsbad64'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)
METRIC_KEYS = ['f1_score', 'auc_prc', 'auc_roc', 'R_AUC_ROC', 'R_AUC_PR', 'VUS_ROC', 'VUS_PR']
COLUMNS = ['method', 'group', 'status', 'seconds', 'run_id', 'commit', 'error'] + METRIC_KEYS

def read_all_results():
    """All result files of all sessions, oldest run first (run_id starts with the UTC start time)."""
    files = sorted(RESULTS_DIR.glob('tsbad64_results*.csv'))
    if not files:
        return pd.DataFrame(columns=COLUMNS)
    frames = [pd.read_csv(f).assign(source_file=f.name) for f in files]
    return pd.concat(frames, ignore_index=True).sort_values('run_id', kind='stable').reset_index(drop=True)
KEEP_MODELS = False   # True also copies CoAD .pth files (~MBs per series) to Drive

def summarize_metrics(metric_file):
    m = json.loads(Path(metric_file).read_text())
    row = {k: m['point_wise'][k] for k in ['f1_score', 'auc_prc', 'auc_roc']}
    row.update(m['vus'])
    return row

def command_for(method, group, timestamp):
    if method == 'CoAD':   # flags from runners/run_all_CoAD.sh (TSB-AD lines)
        return [sys.executable, 'train_classify_mask.py', '--dataset_name', 'TSB-AD', '--group_name', str(group),
                '--model_name', 'freq_tf_v2', '--timestamp', timestamp, '--device', 'cuda:0', '--plot', 'False',
                '--num_epochs', '300', '--batch_size', '128', '--window_multiple', '4', '--lr', '2e-3',
                '--patch_size', '8', '--omega', '10', '--random_seed', '46']
    return [sys.executable, 'baseline_compare.py', '--model_name', method,   # runners/run_dm_baseline.sh
            '--dataset_name', 'TSB-AD', '--group_name', str(group)]

def run_one(method, group):
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run_id = f'{timestamp}_{method}_TSB-AD_{group}'
    out = RUNS / run_id
    out.mkdir(parents=True)
    cmd = command_for(method, group, timestamp)
    (out / 'command.json').write_text(json.dumps(cmd), encoding='utf-8')
    started = time.perf_counter()
    with (out / 'console.log').open('w', encoding='utf-8') as log:
        result = subprocess.run(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
    row = {'method': method, 'group': group, 'run_id': run_id, 'commit': commit,
           'seconds': round(time.perf_counter() - started, 1)}
    try:
        assert result.returncode == 0, f'exit code {result.returncode}'
        g3 = str(group).zfill(3)
        if method == 'CoAD':
            src = REPO / 'mask_results' / 'TSB-AD' / f'freq_tf_v2_max_{timestamp}'
            ignore = None if KEEP_MODELS else shutil.ignore_patterns('*.pth')
            shutil.copytree(src, out / 'artifacts', ignore=ignore)
            shutil.rmtree(src)   # free local disk
            metric_file = next((out / 'artifacts' / 'all' / 'data').glob('group_*.json'))
        else:
            src = REPO / 'baseline_result' / 'TSB-AD' / method
            for rel in [f'data/group_{g3}.json', f'anomaly_score/group_{g3}.npy']:
                (out / 'artifacts' / rel).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src / rel, out / 'artifacts' / rel)
            metric_file = out / 'artifacts' / f'data/group_{g3}.json'
        row.update(summarize_metrics(metric_file), status='success', error='')
    except Exception as exc:
        lines = (out / 'console.log').read_text(encoding='utf-8', errors='replace').strip().splitlines()
        row.update(status='failed', error=f'{exc} | ' + ' / '.join(lines[-3:]))
    pd.DataFrame([row]).reindex(columns=COLUMNS).to_csv(
        RESULTS, mode='a', header=not RESULTS.exists(), index=False)
    return row

In [ ]:
res = read_all_results()
latest = res.drop_duplicates(['method', 'group'], keep='last')
summary = latest.groupby('method').agg(
    success=('status', lambda s: (s == 'success').sum()),
    failed=('status', lambda s: (s == 'failed').sum()),
    median_seconds=('seconds', 'median'),
    max_seconds=('seconds', 'max'),
)
display(summary)
errors = latest.query("status == 'failed'")
if len(errors):
    display(errors.groupby('method')['error'].first().str[-200:].to_frame())

,success,failed,median_seconds,max_seconds
method,,,,
CoAD,64,0,232.35,1141.4
Kmeans,64,0,30.60,171.6
KshapeAD,64,0,87.05,253.0
MP,62,0,30.70,126.2
PCA,64,0,10.00,72.4
SAND,40,0,245.30,2221.1
Series2Graph,26,0,24.45,110.7
damp,62,2,122.25,574.2


,error
method,
damp,"exit code 1 | File ""/content/CoAD/evaluation..."


## 23.5. The 64 TSB-AD series (manifest)

Groups are read from the TSB-AD loops of both runners (must be the same 64). The manifest is written once, before any run, to `eda/manifest_tsbad64.csv`.

In [ ]:
def tsbad_groups(runner_name):
    text = (REPO / 'runners' / runner_name).read_text(encoding='utf-8')
    groups = []
    for block in text.split('for g in $(seq ')[1:]:
        start, end = map(int, re.match(r'(\d+) (\d+)\)', block).groups())
        if '--dataset_name TSB-AD' in block.split('done')[0]:
            groups += list(range(start, end + 1))
    return groups

GROUPS = tsbad_groups('run_all_CoAD.sh')
assert GROUPS == tsbad_groups('run_dm_baseline.sh') and len(GROUPS) == 64, 'Runners disagree on the TSB-AD groups'

manifest_file = STORE / 'eda' / 'manifest_tsbad64.csv'
if not manifest_file.exists():
    rows = []
    for g in GROUPS:
        name = RAW_FILES[g - 1]
        n_points = len(pd.read_csv(RAW / name).dropna())
        rows.append({'dataset': 'TSB-AD', 'group': g, 'source_file': name, 'family': name.split('_')[1],
                     'n_train': int(name.split('.')[0].split('_')[-3]), 'n_test': n_points,
                     'selection_reason': 'Full TSB-AD selection of Table 1 (runner loops)', 'commit': commit})
    pd.DataFrame(rows).to_csv(manifest_file, index=False)
MANIFEST = pd.read_csv(manifest_file)
display(MANIFEST.groupby('family').agg(series=('group', 'count'), test_points=('n_test', 'sum')))

,series,test_points
family,,
IOPS,17,1237469
MGAB,9,880000
SED,3,69997
SVDB,20,3427600
TODS,15,75000


## 23.6. Choose the work for this session

`RUN_METHODS`: e.g. `['CoAD']` or `['MP']` on a GPU runtime, the other six baselines on a CPU runtime. Start with a short family (TODS) to measure run times. `TIME_BUDGET_MIN`: no new run starts after this.

In [ ]:
METHODS = ['CoAD', 'Kmeans', 'KshapeAD', 'SAND', 'Series2Graph', 'MP', 'PCA', 'damp']
SESSION_TAG = 'cpu-a'         # unique per notebook copy running at the same time, e.g. 'cpu-a', 'cpu-b', 'gpu-a'
RESULTS = RESULTS_DIR / f'tsbad64_results_{SESSION_TAG}.csv'   # this session writes only here
RUN_METHODS = ['PCA', 'Kmeans', 'damp', 'Series2Graph']   # methods to run in this session
# RUN_METHODS = ['MP'] #GPU
# RUN_METHODS = ['SAND', 'KshapeAD'] #long time run
RUN_GROUPS = GROUPS           # e.g. MANIFEST.query("family == 'TODS'").group.tolist() for a short first batch
TIME_BUDGET_MIN = 150         # do not start a new run after this many minutes
print('Session', SESSION_TAG, '| writes to', RESULTS.name, '| runs', RUN_METHODS)

Session cpu-a | writes to tsbad64_results_cpu-a.csv | runs ['PCA', 'Kmeans', 'damp', 'Series2Graph']


In [ ]:
print(subprocess.check_output([sys.executable, '-c', 'import sklearn; print(sklearn.__version__)'], text=True))

1.6.1



## 23.7. Run the batch (re-run to continue)

Skips (method, group) pairs that already succeeded; failed ones are retried. One line per run.

In [ ]:
def finished_pairs():
    done = read_all_results().query("status == 'success'")   # finished by any session
    return set(zip(done['method'], done['group'].astype(int)))

todo = [(m, g) for m in RUN_METHODS for g in RUN_GROUPS if (m, g) not in finished_pairs()]
print(f'{len(todo)} runs to do in this session')
session_start = time.perf_counter()
for i, (method, group) in enumerate(todo, 1):
    if (time.perf_counter() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell in a new session to continue.')
        break
    started_at = datetime.now(timezone.utc).strftime('%H:%M UTC')
    print(f'[{i}/{len(todo)}] {method:12s} group {group}: running (started {started_at}) ...', flush=True)
    row = run_one(method, group)
    scores = '  '.join(f"{name}={row.get(key, float('nan')):.4f}" for key, name in
                       [('f1_score', 'F1'), ('auc_prc', 'AUC-PR'), ('R_AUC_PR', 'R-AUC-PR'), ('VUS_PR', 'VUS-PR')])
    print(f"[{i}/{len(todo)}] {method:12s} group {group}: {row['status']:7s} {row['seconds']:7.0f}s  {scores}")

40 runs to do in this session
[1/40] damp         group 242: running (started 13:19 UTC) ...
[1/40] damp         group 242: success     355s  F1=0.4660  AUC-PR=0.4318  R-AUC-PR=0.3740  VUS-PR=0.3686
[2/40] damp         group 247: running (started 13:25 UTC) ...
[2/40] damp         group 247: success     120s  F1=0.0880  AUC-PR=0.0346  R-AUC-PR=0.1192  VUS-PR=0.1067
[3/40] Series2Graph group 251: running (started 13:27 UTC) ...
[3/40] Series2Graph group 251: success      91s  F1=0.8692  AUC-PR=0.9311  R-AUC-PR=0.9885  VUS-PR=0.9782
[4/40] Series2Graph group 252: running (started 13:29 UTC) ...
[4/40] Series2Graph group 252: success      76s  F1=0.5783  AUC-PR=0.5934  R-AUC-PR=0.7983  VUS-PR=0.7698
[5/40] Series2Graph group 253: running (started 13:30 UTC) ...
[5/40] Series2Graph group 253: success      43s  F1=0.8260  AUC-PR=0.8847  R-AUC-PR=0.9261  VUS-PR=0.9111
[6/40] Series2Graph group 254: running (started 13:31 UTC) ...
[6/40] Series2Graph group 254: success      76s  F1=0.1321  AU

## 23.8. Progress and comparison with the paper (all 4 metrics)

Paper = Table 1, TSB-AD columns (%). Delta and verdict (|Delta| <= 3 match, <= 8 partial, else no match) are computed **only for methods that finished all 64 series**; otherwise N/A. Fix the thresholds in `verdict` before looking at results.

In [ ]:
PAPER = pd.DataFrame({   # Table 1, TSB-AD columns, percent
    'F1':       {'CoAD': 49.13, 'Kmeans': 41.42, 'KshapeAD': 35.51, 'SAND': 34.74, 'Series2Graph': 33.48, 'MP': 35.10, 'PCA': 32.49, 'damp': 17.44},
    'AUC-PR':   {'CoAD': 43.66, 'Kmeans': 37.26, 'KshapeAD': 32.67, 'SAND': 31.52, 'Series2Graph': 30.11, 'MP': 27.93, 'PCA': 27.59, 'damp': 11.59},
    'R-AUC-PR': {'CoAD': 40.50, 'Kmeans': 37.64, 'KshapeAD': 31.87, 'SAND': 31.85, 'Series2Graph': 30.13, 'MP': 29.88, 'PCA': 23.88, 'damp': 12.98},
    'VUS-PR':   {'CoAD': 39.83, 'Kmeans': 36.81, 'KshapeAD': 31.27, 'SAND': 31.05, 'Series2Graph': 29.57, 'MP': 28.94, 'PCA': 23.85, 'damp': 12.60},
})
METRIC_NAMES = {'f1_score': 'F1', 'auc_prc': 'AUC-PR', 'R_AUC_PR': 'R-AUC-PR', 'VUS_PR': 'VUS-PR'}

def verdict(delta):
    if pd.isna(delta):
        return 'N/A'
    return 'match' if abs(delta) <= 3 else 'partial' if abs(delta) <= 8 else 'no match'

results = read_all_results()
ok = results.query("status == 'success'").drop_duplicates(['method', 'group'], keep='last')
coverage = ok.groupby('method')['group'].nunique().reindex(METHODS, fill_value=0)
print('Series finished per method (of 64):'); print(coverage.to_string())
latest = results.drop_duplicates(['method', 'group'], keep='last')   # latest attempt of each pair
failed = latest.query("status == 'failed'")                           # still failing (later successes excluded)
if len(failed):
    print(f'\n{len(failed)} series whose latest attempt failed:')
    display(failed[['method', 'group', 'source_file', 'error']])

ours = ok.groupby('method')[list(METRIC_NAMES)].mean().mul(100).rename(columns=METRIC_NAMES)
complete = coverage[coverage == len(GROUPS)].index
rows = []
for method in METHODS:
    for metric in METRIC_NAMES.values():
        our = ours.loc[method, metric] if method in ours.index else np.nan
        delta = our - PAPER.loc[method, metric] if method in complete else np.nan
        rows.append({'method': method, 'metric': metric, 'series': int(coverage[method]),
                     'ours': our, 'paper': PAPER.loc[method, metric], 'delta': delta, 'verdict': verdict(delta)})
comparison = pd.DataFrame(rows)
comparison.to_csv(STORE / 'reports' / 'tsbad64_vs_paper.csv', index=False)
display(comparison.pivot(index='method', columns='metric', values=['ours', 'paper', 'delta']).round(2).reindex(METHODS))
display(comparison.pivot(index='method', columns='metric', values='verdict').reindex(METHODS))

by_family = ok.merge(MANIFEST[['group', 'family']], on='group').groupby(['method', 'family'])['VUS_PR'].mean().mul(100)
display(by_family.unstack().round(2).reindex(METHODS))

Series finished per method (of 64):
method
CoAD            64
Kmeans          64
KshapeAD        64
SAND            64
Series2Graph    64
MP              64
PCA             64
damp            64


ours                         paper                         \
metric       AUC-PR     F1 R-AUC-PR VUS-PR AUC-PR     F1 R-AUC-PR VUS-PR   
method                                                                     
CoAD          40.43  45.99    42.73  41.70  43.66  49.13    40.50  39.83   
Kmeans        41.06  45.18    41.95  41.01  37.26  41.42    37.64  36.81   
KshapeAD      36.12  38.01    36.65  35.86  32.67  35.51    31.87  31.27   
SAND          31.54  34.72    31.82  31.06  31.52  34.74    31.85  31.05   
Series2Graph  33.07  36.10    32.03  31.44  30.11  33.48    30.13  29.57   
MP            28.22  34.94    30.47  29.48  27.93  35.10    29.88  28.94   
PCA           27.72  32.48    23.89  23.92  27.59  32.49    23.88  23.85   
damp          12.14  18.44    13.29  12.99  11.59  17.44    12.98  12.60   

              delta                        
metric       AUC-PR    F1 R-AUC-PR VUS-PR  
method                                     
CoAD          -3.23 -3.14     2.23   1.87  
Kmeans         3.80  3.76     4.31   4.20  
KshapeAD       3.45  2.50     4.78   4.59  
SAND           0.02 -0.02    -0.03   0.01  
Series2Graph   2.96  2.62     1.90   1.87  
MP             0.29 -0.16     0.59   0.54  
PCA            0.13 -0.01     0.01   0.07  
damp           0.55  1.00     0.31   0.39

metric,AUC-PR,F1,R-AUC-PR,VUS-PR
method,,,,
CoAD,partial,partial,match,match
Kmeans,partial,partial,partial,partial
KshapeAD,partial,match,partial,partial
SAND,match,match,match,match
Series2Graph,match,match,match,match
MP,match,match,match,match
PCA,match,match,match,match
damp,match,match,match,match


family,IOPS,MGAB,SED,SVDB,TODS
method,,,,,
CoAD,32.50,41.71,93.19,56.09,22.62
Kmeans,32.52,0.99,92.47,62.52,35.67
KshapeAD,3.39,0.48,87.26,70.88,36.91
SAND,4.04,0.46,90.47,54.45,36.98
Series2Graph,19.66,0.43,14.90,54.53,35.89
MP,4.02,34.50,73.08,37.47,35.97
PCA,21.64,0.40,3.26,44.93,16.75
damp,3.48,0.45,9.61,18.36,24.82


In [ ]:
res = read_all_results()
mp_latest = res[res.method == 'MP'].drop_duplicates('group', keep='last')
done = set(mp_latest.query("status == 'success'")['group'].astype(int))
missing = sorted(set(GROUPS) - done)
print('MP success:', len(done), '/ 64')
print('Never run or not finished:', [g for g in missing if g not in set(mp_latest['group'].astype(int))])
display(mp_latest.query("status != 'success'")[['group', 'source_file', 'run_id', 'error']])

MP success: 62 / 64
Never run or not finished: [249, 253]


,group,source_file,run_id,error


## 24. M6-B: the authors' ablation variants (Table 2)

Variants already in the repository (`--model_name` of `train_classify_mask.py`, listed in
`runners/run_ablation_CoAD.sh`), run on the same 64 series with **exactly the CoAD command of 23.4**: only
`--model_name` changes (plus the `--window_multiple` the runner gives some variants). Two differences from the
authors' runner, both on purpose: `--dataset_name TSB-AD` (the runner's `TSB-AD-U` is not accepted by
`load_dataset`) and batch 128 (the runner uses 64; 128 keeps the comparison with our CoAD runs controlled).

Recommended: `time_soft` (no frequency branch), `guide_hard` (hard instead of soft mask), `freq_time_mean` (mean
instead of max fusion); about 5.3 h of T4 each. Run one variant per notebook copy, each with its own `ABL_TAG`.
Needs 23.1-23.5 in the same runtime.

### 24.1. Settings and one-run helper

In [ ]:
ABLATION_VARIANTS = {   # model_name -> (Table 2 row, extra flags of runners/run_ablation_CoAD.sh)
    'time_soft':        ('OE (Time) + MAE (Guide w/ Soft Mask)', []),
    'guide_hard':       ('OE+MAE (Guide w/ Hard Mask)', ['--window_multiple', '4']),
    'freq_time_mean':   ('Decision_Mean', []),
    'freq_hard':        ('OE+MAE (Random)', []),
    'freq_gating':      ('OE+MAE (Grating)', []),
    'freq_time_add':    ('Feature_Add', []),
    'freq_time_gate':   ('Feature_Gate', []),
    'freq_time_point':  ('Step Level', ['--window_multiple', '4']),
    'freq_time_window': ('Window Level', ['--window_multiple', '1']),
}
# Table 2, TSB-AD columns (%). Rows of the 'Cooperative' block matched by row order (check against the PDF).
PAPER_T2 = pd.DataFrame({
    'CoAD':             [49.13, 43.66, 40.50, 39.83],
    'time_soft':        [43.10, 36.87, 37.91, 36.92],
    'guide_hard':       [44.06, 38.57, 35.31, 34.76],
    'freq_time_mean':   [42.24, 36.23, 33.71, 33.18],
    'freq_hard':        [45.01, 39.31, 35.79, 35.20],
    'freq_gating':      [47.07, 41.48, 37.39, 37.08],
    'freq_time_add':    [40.11, 35.15, 33.40, 32.94],
    'freq_time_gate':   [42.23, 37.20, 34.87, 34.40],
    'freq_time_point':  [42.52, 36.06, 34.02, 33.37],
    'freq_time_window': [22.43, 17.08, 15.71, 15.53],
}, index=['F1', 'AUC-PR', 'R-AUC-PR', 'VUS-PR']).T

ABL_RUNS = STORE / 'runs' / 'ablation'
ABL_RUNS.mkdir(parents=True, exist_ok=True)
ABL_COLUMNS = ['variant', 'group', 'status', 'seconds', 'run_id', 'commit', 'error'] + METRIC_KEYS

def read_ablation():
    files = sorted(RESULTS_DIR.glob('ablation_results*.csv'))
    if not files:
        return pd.DataFrame(columns=ABL_COLUMNS)
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True).sort_values('run_id', kind='stable')

def run_variant(variant, group, results_file):
    """Same command as the CoAD runs of 23.4; only --model_name (and the runner's extra flags) change."""
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run_id = f'{timestamp}_{variant}_TSB-AD_{group}'
    out = ABL_RUNS / run_id
    out.mkdir(parents=True)
    cmd = command_for('CoAD', group, timestamp)
    cmd[cmd.index('--model_name') + 1] = variant
    cmd += ABLATION_VARIANTS[variant][1]   # argparse keeps the last value of a repeated flag
    (out / 'command.json').write_text(json.dumps(cmd), encoding='utf-8')
    started = time.perf_counter()
    with (out / 'console.log').open('w', encoding='utf-8') as log:
        result = subprocess.run(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
    row = {'variant': variant, 'group': group, 'run_id': run_id, 'commit': commit,
           'seconds': round(time.perf_counter() - started, 1)}
    try:
        assert result.returncode == 0, f'exit code {result.returncode}'
        src = REPO / 'mask_results' / 'TSB-AD' / f'{variant}_max_{timestamp}'
        shutil.copytree(src, out / 'artifacts', ignore=None if KEEP_MODELS else shutil.ignore_patterns('*.pth'))
        shutil.rmtree(src)
        metric_file = next((out / 'artifacts' / 'all' / 'data').glob('group_*.json'))
        row.update(summarize_metrics(metric_file), status='success', error='')
    except Exception as exc:
        lines = (out / 'console.log').read_text(encoding='utf-8', errors='replace').strip().splitlines()
        row.update(status='failed', error=f'{exc} | ' + ' / '.join(lines[-3:]))
    pd.DataFrame([row]).reindex(columns=ABL_COLUMNS).to_csv(
        results_file, mode='a', header=not results_file.exists(), index=False)
    return row

### 24.2. Run (GPU; re-run to continue)

In [ ]:
ABL_TAG = 'gpu-a'                 # different in each notebook copy running at the same time
RUN_VARIANTS = ['time_soft']      # one variant per copy: 'time_soft' / 'guide_hard' / 'freq_time_mean'
RUN_GROUPS_ABL = GROUPS           # all 64; MANIFEST.query("family == 'TODS'").group.tolist() for a quick first test
TIME_BUDGET_MIN = 150
ABL_RESULTS = RESULTS_DIR / f'ablation_results_{ABL_TAG}.csv'

done = read_ablation().query("status == 'success'")
done = set(zip(done['variant'], done['group'].astype(int)))
todo = [(v, g) for v in RUN_VARIANTS for g in RUN_GROUPS_ABL if (v, g) not in done]
print(f'{len(todo)} runs to do | writes to {ABL_RESULTS.name}')
session_start = time.perf_counter()
for i, (variant, group) in enumerate(todo, 1):
    if (time.perf_counter() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue.')
        break
    row = run_variant(variant, group, ABL_RESULTS)
    print(f"[{i}/{len(todo)}] {variant} group {group}: {row['status']} {row['seconds']:.0f}s "
          f"VUS-PR={row.get('VUS_PR', float('nan')):.4f}", flush=True)

### 24.3. Against CoAD and Table 2

Our CoAD already differs from the paper by a few points (M4), so the comparison is on the **change** relative to
CoAD: `ours - CoAD` is computed on the same series (our CoAD run of 23.7), `paper - CoAD` from Table 2.
`same direction` = the variant moves the metric the same way as in the paper. Rows with `series` < 64 are partial.

In [ ]:
NAMES = {'f1_score': 'F1', 'auc_prc': 'AUC-PR', 'R_AUC_PR': 'R-AUC-PR', 'VUS_PR': 'VUS-PR'}
abl = read_ablation().query("status == 'success'").drop_duplicates(['variant', 'group'], keep='last')
coad = (read_all_results().query("method == 'CoAD' and status == 'success'")
        .drop_duplicates('group', keep='last').set_index('group'))
rows = []
for variant, runs in abl.groupby('variant'):
    groups = runs['group'].astype(int).tolist()
    ours = 100 * runs.set_index('group')[list(NAMES)].mean()
    base = 100 * coad.loc[groups, list(NAMES)].mean()
    for key, name in NAMES.items():
        rows.append({'variant': variant, 'Table 2 row': ABLATION_VARIANTS[variant][0], 'series': len(groups),
                     'metric': name, 'ours': ours[key], 'ours - CoAD': ours[key] - base[key],
                     'paper': PAPER_T2.loc[variant, name],
                     'paper - CoAD': PAPER_T2.loc[variant, name] - PAPER_T2.loc['CoAD', name]})
m6b = pd.DataFrame(rows)
m6b['same direction'] = np.sign(m6b['ours - CoAD']) == np.sign(m6b['paper - CoAD'])
(STORE / 'reports').mkdir(exist_ok=True)
m6b.to_csv(STORE / 'reports' / 'm6b_ablation.csv', index=False)
display(m6b.round(2))